# DP-OT quick diagnostics

A fast read of the two **non-private** diagnostics, with **no GNN training**:

- **Concept-shift probe (#2)** — `dp_ot/eval/diagnostics.py`: how much of the target
  gap is irreducible P(Y|X) shift (vs. structure) that importance weighting can't fix.
- **FGW alignment (#1)** — `dp_ot/adapt/fgw_align.py`, prototype-level read: does
  frame misalignment explain the residual gap?

For the full AUROC verdict (`oracle_fgw` vs `oracle` vs `source_only`, with the
held-out `target_oracle`), run `colab_real_data.ipynb` instead — that one trains GNNs
and is slow. This notebook is for quickly reading the diagnostics.

## 0. Clone + install

In [ ]:
import os
REPO_URL = "https://github.com/ChefAltoids/MM-edgeDP"
BRANCH   = "dp-ot"
REPO_DIR = "/content/MM-edgeDP"

if not os.path.isdir(REPO_DIR):
    !git clone --branch $BRANCH $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git fetch origin && git checkout $BRANCH && git pull --ff-only
os.chdir(REPO_DIR)

# Diagnostics need torch only to LOAD the graphs (no GNN training here).
!pip install -q torch_geometric ogb POT
print("Ready.")

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)
import numpy as np

from dp_ot.eval.diagnostics import concept_shift_probe, print_concept_shift
from dp_ot.adapt.prototypes import compute_target_summaries, fit_public_prototypes
from dp_ot.adapt.fgw_align import fit_target_prototypes, fgw_aligned_target_mass, coupling_diagnostics
from dp_ot.adapt.dp_histogram import nonprivate_histogram


def fgw_quickread(G_source, G_target, K=64, d_max=20, B=5.0, fgw_alpha=0.5, seed=0):
    """Prototype-level FGW alignment read (no GNN). Compares the FGW-aligned target
    mass (independent target frame + geometric correspondence) against the shared-
    frame mass (target nodes assigned to SOURCE centroids)."""
    ss = compute_target_summaries(G_source, d_max=d_max, B=B)
    cents, _, _ = fit_public_prototypes(ss, K, seed=seed)
    ts = compute_target_summaries(G_target, d_max=d_max, B=B)
    shared = nonprivate_histogram(ts, cents)                      # shared-frame target mass
    cents_t, alpha_t = fit_target_prototypes(ts, K, seed=seed)
    aligned, T, cost = fgw_aligned_target_mass(cents, cents_t, alpha_t,
                                               fusion_alpha=fgw_alpha, seed=seed)
    diag = coupling_diagnostics(T)
    l1 = float(np.abs(aligned - shared).sum())
    print(f"FGW correspondence cost             : {cost:.4f}   (lower = cleaner geometric match)")
    print(f"coupling peak / row-entropy         : {diag['coupling_mean_peak']:.3f} / {diag['coupling_mean_row_entropy']:.3f}")
    print(f"L1(FGW-aligned vs shared-frame mass): {l1:.4f}")
    if l1 < 0.15:
        print("  -> ~aligned: FGW recovers essentially the mass the shared frame already had")
        print("     (little misindexing). Points to concept shift, not misalignment.")
    else:
        print("  -> FGW reindexes target mass differently; check the oracle_fgw AUROC row")
        print("     in colab_real_data.ipynb to see whether that actually recovers gain.")
    return dict(cost=cost, l1=l1, **diag)

print("helpers ready")

## 1. OGB-arxiv (auto-downloads)

In [ ]:
from dp_ot.data.real_splits import load_ogb_arxiv_temporal
G_source, G_target = load_ogb_arxiv_temporal(source_before_year=2018, target_from_year=2018, root="data/ogb")
print(f"source {G_source.num_nodes} nodes | target {G_target.num_nodes} nodes\n")

print("--- concept-shift probe (#2) ---")
print_concept_shift(concept_shift_probe(G_source, G_target, seed=0))
print("\n--- FGW alignment (#1), prototype-level ---")
_ = fgw_quickread(G_source, G_target, K=64, d_max=20, B=5.0)

## 2. ACM↔DBLP (upload the .mat files)

ACMv9 / DBLPv7 are not pip-installable — provide download URLs, gdown, or upload
`acmv9.mat` / `dblpv7.mat` into `GDA_ROOT`.

In [ ]:
import os
GDA_ROOT = 'data/graph_da'
os.makedirs(GDA_ROOT, exist_ok=True)
ACM_URL  = None   # e.g. 'https://.../acmv9.mat' (loader fetches if set)
DBLP_URL = None
# Or upload manually:
# from google.colab import files; files.upload()   # then move .mat into GDA_ROOT
print('Files in', GDA_ROOT, '->', os.listdir(GDA_ROOT))

In [ ]:
from dp_ot.data.real_splits import load_graph_da_pair
G_s, G_t = load_graph_da_pair('acmv9', 'dblpv7', root=GDA_ROOT,
                              source_url=ACM_URL, target_url=DBLP_URL)
print(f"ACMv9 {G_s.num_nodes} nodes | DBLPv7 {G_t.num_nodes} nodes | "
      f"{G_s.x.shape[1]} feats | {int(G_s.y.max())+1} classes\n")

print("--- concept-shift probe (#2) ---")
print_concept_shift(concept_shift_probe(G_s, G_t, seed=0))
print("\n--- FGW alignment (#1), prototype-level ---")
_ = fgw_quickread(G_s, G_t, K=64, d_max=20, B=5.0)